<a href="https://colab.research.google.com/github/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%205/5.3%20Task%20Flow%2C%20Memory%2C%20and%20Evaluation/5.3.4%20Tutorial_%20Logging%2C%20Validation%2C%20and%20Handling%20Failure%20States%20Openrouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Tutorial: Logging, Validation, and Handling Failure States

In this tutorial, we'll add simple logging to track what our agents do and
handle failures gracefully. This is what makes agents ready for real-world use.

Learning Objectives:
- Add simple logging to track agent operations
- Handle agent failures gracefully
- Create basic monitoring for production use
- Build reliable and debuggable agents

In [1]:
# Install required packages
!pip install -q "pydantic-ai-slim[openrouter]==2.48.0" openai==3.19.0 pydantic==2.13.5 python-dotenv==1.2.3

import os
import logging
from typing import List
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.5/125.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.6/146.6 kB 5.8 MB/s eta 0:00:00


In [2]:
load_dotenv()

False

In [3]:
# OpenRouter setup
from getpass import getpass
from pydantic_ai.models.openrouter import OpenRouterModel
from pydantic_ai.providers.openrouter import OpenRouterProvider

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
MODEL = "openai/gpt-4.1-mini"

# PydanticAI model routed through OpenRouter; used by every agent below.
openrouter_model = OpenRouterModel(MODEL, provider=OpenRouterProvider(api_key=OPENROUTER_API_KEY))

Enter your OpenRouter API key: ··········


### Simple Logging Setup

Let's set up basic Python logging.

In [4]:
# Set up simple logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('agent.log'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('agent')

In [5]:
def log_action(action: str, success: bool = True):
    """Simple logging function."""
    if success:
        logger.info(f"✅ {action}")
        print(f"✅ {action}")
    else:
        logger.error(f"❌ {action}")
        print(f"❌ {action}")

### Agent with Logging

Let's create an agent that logs everything it does.

In [6]:
class LoggedResponse(BaseModel):
    answer: str = Field(description="The main answer")
    actions_logged: int = Field(description="Number of actions logged")

logged_agent = Agent(openrouter_model, output_type=LoggedResponse)


In [7]:
@logged_agent.tool
async def logged_search(ctx: RunContext[None], query: str) -> str:
    """Search tool with logging."""
    log_action(f"Starting search for: {query}")

    try:
        import random
        if random.random() < 0.3:
            raise Exception("Search timeout")

        log_action(f"Search completed for: {query}")
        return f"Search results for: {query}"
    except Exception as e:
        log_action(f"Search failed for {query}: {e}", success=False)
        return f"Search unavailable for: {query}"

In [8]:
@logged_agent.tool
async def logged_calculation(ctx: RunContext[None], expression: str) -> str:
    """Calculator with logging."""
    log_action(f"Calculating: {expression}")

    try:
        result = eval(expression)
        log_action(f"Calculation successful: {expression} = {result}")
        return f"{expression} = {result}"
    except Exception as e:
        log_action(f"Calculation failed: {expression} - {e}", success=False)
        return f"Cannot calculate: {expression}"

### Failure Tracking

Let's track failures for monitoring.

In [9]:
failures = []

def track_failure(operation: str, error: str):
    """Track failures for monitoring."""
    failure = {
        'operation': operation,
        'error': error,
        'count': len(failures) + 1
    }
    failures.append(failure)
    log_action(f"Failure #{failure['count']}: {operation} - {error}", success=False)


In [10]:
@logged_agent.tool
async def monitored_operation(ctx: RunContext[None], task: str) -> str:
    """Operation with failure monitoring."""
    log_action(f"Starting {task}")

    try:
        import random
        if random.random() < 0.4:
            raise Exception(f"{task} service down")

        log_action(f"Completed {task}")
        return f"Successfully completed: {task}"
    except Exception as e:
        track_failure(task, str(e))
        return f"Failed to complete {task}, using fallback"


In [11]:
tests = [
    "Search for AI trends",
    "Calculate 20 + 30",
    "Calculate 10 / 0",
    "Process data analysis",
    "Generate report"
]

for test in tests:
    print(f"\nTest: {test}")
    try:
        result = await logged_agent.run(test)
        print(f"Actions logged: {result.output.actions_logged}")
    except Exception as e:
        log_action(f"Agent failed: {e}", success=False)



Test: Search for AI trends
✅ Starting search for: AI trends 2024
✅ Search completed for: AI trends 2024
✅ Starting search for: latest AI trends 2024
✅ Search completed for: latest AI trends 2024


ERROR:agent:❌ Search failed for top AI trends 2024: Search timeout


✅ Starting search for: top AI trends 2024
❌ Search failed for top AI trends 2024: Search timeout
✅ Starting search for: emerging AI trends 2024
✅ Search completed for: emerging AI trends 2024
Actions logged: 7

Test: Calculate 20 + 30
✅ Calculating: 20 + 30
✅ Calculation successful: 20 + 30 = 50
Actions logged: 1

Test: Calculate 10 / 0


ERROR:agent:❌ Calculation failed: 10 / 0 - division by zero


✅ Calculating: 10 / 0
❌ Calculation failed: 10 / 0 - division by zero
Actions logged: 1

Test: Process data analysis


...

✅ Starting data analysis
❌ Failure #1: data analysis - data analysis service down
✅ Starting search for: best practices for data analysis
✅ Search completed for: best practices for data analysis
Actions logged: 3

Test: Generate report


ERROR:agent:❌ Failure #2: generate report - generate report service down


✅ Starting generate report
❌ Failure #2: generate report - generate report service down


ERROR:agent:❌ Search failed for how to generate a report: Search timeout


✅ Starting search for: how to generate a report
❌ Search failed for how to generate a report: Search timeout
Actions logged: 2
